# 中国語チェス台詞 — edge-tts 一括音声生成

1. このノートブックをGoogle Colabで開き、順番にセルを実行します。
2. 2つのJSONをアップロードします（名前は問いません）。
3. `zhHans` が同じセリフは1つだけ生成し、IDごとの対応表 `audio-map.json` を作ります。
4. 標準は **AAC / M4A 32 kbps（iPhone対応）**。最小容量重視なら `FORMAT = "webm"` （Opus / 20 kbps）に変更してください。ブラウザ互換性を優先する場合は M4A 推奨。
5. 完成したZIPをダウンロード。エラーが出た場合は同じ生成セルを再実行して未完了分のみ再試行できます。

音声はオンラインのEdge TTSにアクセスして生成されます。利用条件については別途ご確認ください。

In [ ]:
!pip -q install -U edge-tts

import os, json, asyncio, subprocess, zipfile, hashlib, time, random
from pathlib import Path
from google.colab import files

VOICE = "zh-CN-XiaoxiaoNeural"
RATE = "+0%"
FORMAT = "m4a"  # "m4a" (AAC, iPhone向き) or "webm" (Opus, より軽量)
BITRATE = "32k" if FORMAT == "m4a" else "20k"
CONCURRENCY = 2  # 高負荷やエラーを避けるため控えめに
OUT = Path('/content/chess_tts')
AUDIO = OUT / 'audio'
AUDIO.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()  # JSONファイル2つを選択
entries = []
for filename, raw in uploaded.items():
    if not filename.lower().endswith('.json'):
        continue
    data = json.loads(raw.decode('utf-8-sig'))
    if not isinstance(data, list):
        raise ValueError(f'{filename}: JSONのトップレベルが配列ではありません')
    entries.extend(data)

if not entries:
    raise ValueError('JSONファイルをアップロードしてください')
if any(not isinstance(x, dict) or not x.get('id') or not x.get('zhHans') for x in entries):
    raise ValueError('各要素に id と zhHans が必要です')
if len({x['id'] for x in entries}) != len(entries):
    raise ValueError('重複IDがあります。入力を確認してください。')

# 同じ中国語は1音声を共有する
texts = list(dict.fromkeys(x['zhHans'].strip() for x in entries))
name_for = {s: hashlib.sha256(s.encode('utf-8')).hexdigest()[:16] + '.' + FORMAT for s in texts}
audio_map = {x['id']: 'audio/' + name_for[x['zhHans'].strip()] for x in entries}
(OUT / 'audio-map.json').write_text(json.dumps(audio_map, ensure_ascii=False, indent=2), encoding='utf-8')
(OUT / 'lines-index.json').write_text(json.dumps([
  {'id':x['id'], 'zhHans':x['zhHans'], 'audio':audio_map[x['id']]}
  for x in entries], ensure_ascii=False, indent=2), encoding='utf-8')
print(f'セリフ {len(entries)}件、ユニーク音声 {len(texts)}件、形式 {FORMAT} ({BITRATE})')

In [ ]:
import edge_tts
from tqdm.auto import tqdm

async def generate_one(text, semaphore):
    dest = AUDIO / name_for[text]
    if dest.exists() and dest.stat().st_size > 300:
        return True
    async with semaphore:
        tmp_mp3 = AUDIO / (name_for[text] + '.tmp.mp3')
        tmp_encoded = AUDIO / (name_for[text] + '.tmp.' + FORMAT)
        for attempt in range(5):
            try:
                tmp_mp3.unlink(missing_ok=True)
                tmp_encoded.unlink(missing_ok=True)
                await edge_tts.Communicate(text, VOICE, rate=RATE).save(str(tmp_mp3))
                if not tmp_mp3.exists() or tmp_mp3.stat().st_size < 300:
                    raise RuntimeError('空の音声')
                codec = ['-c:a', 'aac', '-b:a', BITRATE] if FORMAT == 'm4a' else ['-c:a', 'libopus', '-b:a', BITRATE, '-vbr', 'on']
                cp = subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-i', str(tmp_mp3),
                                     '-vn', '-ac', '1', *codec, str(tmp_encoded)], capture_output=True, text=True)
                if cp.returncode != 0:
                    raise RuntimeError(cp.stderr[-600:])
                tmp_encoded.replace(dest)
                return True
            except Exception as e:
                if attempt == 4:
                    print(f'失敗: {text} : {e}')
                    return False
                await asyncio.sleep(min(30, 2 ** attempt + random.random()))
            finally:
                tmp_mp3.unlink(missing_ok=True)
                tmp_encoded.unlink(missing_ok=True)

async def main():
    sem = asyncio.Semaphore(CONCURRENCY)
    pending = [s for s in texts if not (AUDIO / name_for[s]).exists()]
    print('生成対象:', len(pending))
    tasks = [asyncio.create_task(generate_one(s, sem)) for s in pending]
    results = []
    for task in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        results.append(await task)
    print('生成成功:', sum(results), '/ 失敗:', len(results) - sum(results))
    return all(results)

success = await main()  # Colabではセル最上位のawaitが使えます


In [ ]:
missing = [s for s in texts if not (AUDIO / name_for[s]).exists()]
if missing:
    print('未生成が', len(missing), '件あります。前のセルを再実行してください。')
else:
    zip_path = Path('/content/chess-edge-tts-' + FORMAT + '.zip')
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob('*'):
            if p.is_file() and '.tmp.' not in p.name:
                z.write(p, p.relative_to(OUT))
    total_kb = sum(p.stat().st_size for p in AUDIO.glob('*.' + FORMAT)) / 1024
    print(f'音声合計: {total_kb:.1f} KiB / ZIP: {zip_path.stat().st_size/1024:.1f} KiB')
    files.download(str(zip_path))
